In [ ]:
"""import requests
from bs4 import BeautifulSoup

url = "https://www.adsoftheworld.com/highlighted"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

anchors = soup.find_all("a", href=lambda h: h and "/campaigns/" in h and not h.endswith("/new"))
urls = list(set([a["href"] for a in anchors]))  # set removes duplicates

print(f"Found {len(urls)} campaigns")
print(urls[:5])
"""

In [ ]:
"""import requests
from bs4 import BeautifulSoup
import re

url = "https://www.adsoftheworld.com/campaigns/natalie-portman-for-tiffany-amp-co-a-hardwear-film"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

title = soup.select_one("h1").get_text(strip=True)
brand = soup.find("a", href=re.compile(r"/brands/")).get_text(strip=True)
agency = soup.find("a", href=re.compile(r"/agencies/")).get_text(strip=True)
country = soup.find("a", href=re.compile(r"/countries/")).get_text(strip=True)
region = soup.find("a", href=re.compile(r"/regions/")).get_text(strip=True)
medium = soup.find("a", href=re.compile(r"/medium_types/")).get_text(strip=True)
industry = soup.find("a", href=re.compile(r"/industries/")).get_text(strip=True)

# Grab all paragraphs longer than 40 chars, skip nav/footer noise
desc_paragraphs = []
for p in soup.find_all("p"):
    t = p.get_text(strip=True)
    if t and len(t) > 40:
        desc_paragraphs.append(t)
description = "\n\n".join(desc_paragraphs[:5])

print("title:", title)
print("brand:", brand)
print("agency:", agency)
print("country:", country)
print("region:", region)
print("medium:", medium)
print("industry:", industry)
print("description:", description)
"""

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import json
import time
import random
from datetime import datetime

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

# ─────────────────────────────────────────────
# Step 1 - Collect campaign URLs from listing pages
# ─────────────────────────────────────────────

def collect_urls(start_page=2, end_page=10):
    urls = set()
    
    for page_num in range(start_page, end_page + 1):
        print(f"Collecting page {page_num}...")
        response = requests.get(
            f"https://www.adsoftheworld.com/highlighted?page={page_num}",
            headers=headers
        )
        soup = BeautifulSoup(response.text, "html.parser")
        anchors = soup.find_all("a", href=lambda h: h and "/campaigns/" in h and not h.endswith("/new"))
        
        page_urls = set([a["href"] for a in anchors])
        urls.update(page_urls)
        
        print(f"  Page {page_num}: +{len(page_urls)} urls (total {len(urls)})")
        time.sleep(random.uniform(1, 2))  # polite delay between pages
    
    # Convert relative to absolute URLs
    return [f"https://www.adsoftheworld.com{u}" if u.startswith("/") else u for u in urls]


# ─────────────────────────────────────────────
# Step 2 - Scrape a single campaign page
# ─────────────────────────────────────────────

def scrape_campaign(url):
    try:
        response = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(response.text, "html.parser")

        title   = soup.select_one("h1").get_text(strip=True) if soup.select_one("h1") else ""
        brand   = soup.find("a", href=re.compile(r"/brands/")).get_text(strip=True) if soup.find("a", href=re.compile(r"/brands/")) else ""
        agency  = soup.find("a", href=re.compile(r"/agencies/")).get_text(strip=True) if soup.find("a", href=re.compile(r"/agencies/")) else ""
        country = soup.find("a", href=re.compile(r"/countries/")).get_text(strip=True) if soup.find("a", href=re.compile(r"/countries/")) else ""
        #region  = soup.find("a", href=re.compile(r"/regions/")).get_text(strip=True) if soup.find("a", href=re.compile(r"/regions/")) else ""
        medium  = soup.find("a", href=re.compile(r"/medium_types/")).get_text(strip=True) if soup.find("a", href=re.compile(r"/medium_types/")) else ""
        industry = soup.find("a", href=re.compile(r"/industries/")).get_text(strip=True) if soup.find("a", href=re.compile(r"/industries/")) else ""
        tags    = [a.get_text(strip=True) for a in soup.find_all("a", href=re.compile(r"/collections/"))]

        # Description - grab paragraphs longer than 40 chars
        desc_paragraphs = []
        for p in soup.find_all("p"):
            t = p.get_text(strip=True)
            if t and len(t) > 40:
                desc_paragraphs.append(t)
        description = "\n\n".join(desc_paragraphs[:5])

        # Thumbnail from og:image meta tag
        og_img = soup.find("meta", property="og:image")
        thumbnail_url = og_img["content"] if og_img and og_img.get("content") else ""

        # Published date and media count from AOTW summary sentence
        full_text = soup.get_text()
        date_match = re.search(r"published in .+? in ([A-Za-z]+,\s*\d{4})", full_text)
        media_match = re.search(r"contains (\d+) media asset", full_text)

        return {
            "id":             url.rstrip("/").split("/")[-1],
            "url":            url,
            "title":          title,
            "brand":          brand,
            "agency":         agency,
            "country":        country,
            #"region":         region,
            "medium":         medium,
            "industry":       industry,
            "description":    description,
            "tags":           tags,
            "thumbnail_url":  thumbnail_url,
            "published_date": date_match.group(1) if date_match else "",
            "media_count":    int(media_match.group(1)) if media_match else 0,
            "crawled_at":     datetime.utcnow().isoformat(),
        }

    except Exception as e:
        print(f"  Failed: {url} — {e}")
        return None


# ─────────────────────────────────────────────
# Step 3 - Run full pipeline
# ─────────────────────────────────────────────

def run(start_page=2, end_page=10, output="campaigns.json"):
    # Collect all URLs first
    all_urls = collect_urls(start_page=start_page, end_page=end_page)
    print(f"\nTotal unique URLs collected: {len(all_urls)}")

    # Scrape each campaign
    campaigns = []
    for i, url in enumerate(all_urls, 1):
        print(f"[{i}/{len(all_urls)}] Scraping: {url}")
        campaign = scrape_campaign(url)
        if campaign:
            campaigns.append(campaign)
            print(f"  ✓ {campaign['brand']} — {campaign['title'][:60]}")
        
        time.sleep(random.uniform(1, 2))  # polite delay between campaigns

        # Save checkpoint every 20 items
        if i % 20 == 0:
            with open(output, "w", encoding="utf-8") as f:
                json.dump(campaigns, f, ensure_ascii=False, indent=2)
            print(f"  [checkpoint saved — {len(campaigns)} campaigns]")

    # Final save
    with open(output, "w", encoding="utf-8") as f:
        json.dump(campaigns, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Done! {len(campaigns)} campaigns saved to {output}")
    return campaigns


# ─────────────────────────────────────────────
# Run it
# ─────────────────────────────────────────────

campaigns = run(start_page=4, end_page=10, output="campaigns.json")

In [ ]:
import json

with open("campaigns.json", "r", encoding="utf-8") as f:
    campaigns = json.load(f)

# Replace id with numeric index
for i, campaign in enumerate(campaigns, 1):
    campaign["id"] = i

with open("campaigns.json", "w", encoding="utf-8") as f:
    json.dump(campaigns, f, ensure_ascii=False, indent=2)

print(f"Updated {len(campaigns)} campaigns with numeric ids")
print(campaigns[0]) 

In [3]:
import json
with open("campaigns.json", "r", encoding="utf-8") as f:
    campaigns = json.load(f)

print(f"Loaded {len(campaigns)} campaigns")
print(campaigns[0]) 

Loaded 424 campaigns
{'id': 1, 'url': 'https://www.adsoftheworld.com/campaigns/rebranding-and-communication-for-selfish-reinforcing-its-positioning', 'title': 'Rebranding and communication for Selfish, reinforcing its positioning', 'brand': 'Selfish', 'agency': 'LV Agency', 'country': 'Portugal', 'medium': '360°', 'industry': 'Travel and Tourism', 'description': 'Rebranding and communication for Selfish, reinforcing its positioning\n\nSelfish is one of the brands within the Starfoods group, with its own positioning in the restaurant industry, focused on fish, freshness, and flavor.\n\nAt LV, we recently supported the brand through a refresh process aimed at clearly reinforcing the freshness of its fish. The work included updating the visual identity, simplifying the brand language, and strengthening its positioning, making the brand more contemporary, direct, and aligned with its audience.\n\nWe maintained the irreverence that has always defined Selfish, but introduced a cleaner, more 

In [4]:
from rank_bm25 import BM25Okapi

# Combine fields into one string per campaign for BM25
def build_corpus(campaigns):
    corpus = []
    for c in campaigns:
        text = f"{c['title']} {c['brand']} {c['agency']} {c['country']} {c['industry']} {c['medium']} {c['description']}"
        tokens = text.lower().split()  # BM25 works on token list
        corpus.append(tokens)
    return corpus

corpus = build_corpus(campaigns)
bm25 = BM25Okapi(corpus)

print(f"BM25 index built with {len(corpus)} documents")

BM25 index built with 424 documents


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load pre-trained embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Combine fields into one string per campaign for embedding
def build_semantic_corpus(campaigns):
    texts = []
    for c in campaigns:
        text = f"{c['title']}. {c['brand']}. {c['industry']}. {c['description']}"
        texts.append(text)
    return texts

texts = build_semantic_corpus(campaigns)

# Embed all campaigns — this may take a minute
print("Embedding campaigns...")
embeddings = model.encode(texts, show_progress_bar=True)

# Save embeddings to disk so you don't have to re-run every time
#np.save("embeddings.npy", embeddings)

print(f"Embeddings shape: {embeddings.shape}")  # should be (num_campaigns, 384)

c:\Users\khanh\ai-engineering-fordham\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [6]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

# Load campaigns
with open("campaigns.json", "r", encoding="utf-8") as f:
    campaigns = json.load(f)

# Load embeddings
embeddings = np.load("embeddings.npy")

# Rebuild BM25 index (no file to save/load, rebuilds instantly from campaigns)
def build_corpus(campaigns):
    corpus = []
    for c in campaigns:
        text = f"{c['title']} {c['brand']} {c['agency']} {c['country']} {c['industry']} {c['medium']} {c['description']}"
        tokens = text.lower().split()
        corpus.append(tokens)
    return corpus

corpus = build_corpus(campaigns)
bm25 = BM25Okapi(corpus)

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"Campaigns: {len(campaigns)}")
print(f"Embeddings shape: {embeddings.shape}")
print("BM25 index ready")
print("Model loaded")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 386.96it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Campaigns: 424
Embeddings shape: (424, 384)
BM25 index ready
Model loaded


In [28]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Load embeddings if not already in memory
embeddings = np.load("embeddings.npy")

def hybrid_search(query, campaigns, bm25, embeddings, model, top_k=5, bm25_weight=0.5, semantic_weight=0.5):
    # BM25 scores
    query_tokens = query.lower().split()
    bm25_scores = bm25.get_scores(query_tokens)

    # Semantic scores
    query_embedding = model.encode([query])
    semantic_scores = np.dot(embeddings, query_embedding.T).flatten()  # cosine-like similarity

    # Normalize both scores to 0-1 range so they are comparable
    bm25_scores_norm = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-9)
    semantic_scores_norm = (semantic_scores - semantic_scores.min()) / (semantic_scores.max() - semantic_scores.min() + 1e-9)

    # Combine
    final_scores = bm25_weight * bm25_scores_norm + semantic_weight * semantic_scores_norm

    # Get top K indices
    top_indices = np.argsort(final_scores)[::-1][:top_k]

    # Return results
    results = []
    for idx in top_indices:
        results.append({
            "score": round(float(final_scores[idx]), 4),
            "id": campaigns[idx]["id"],
            "title": campaigns[idx]["title"],
            "brand": campaigns[idx]["brand"],
            "industry": campaigns[idx]["industry"],
            "description": campaigns[idx]["description"], 
            "url": campaigns[idx]["url"],
        })
    return results


# Test it
results = hybrid_search("Adidas", campaigns, bm25, embeddings, model, top_k=5)

for r in results:
    print(f"[{r['score']}] {r['brand']} — {r['title']}")
    print(f"  {r['description']}\n")

[0.5] Salomon — Shaping New Futures
  On February 1, SALOMON launches Shaping New Futures, its latest global brand campaign, positioning innovation as the brand’s cultural engine: a way to shape sport, influence style, and design a modern mountain sport culture, while staying true to Salomon’s roots.

Over the past three years, SALOMON has followed a clear and purposeful brand trajectory. In 2024, Welcome Back to Earth reaffirmed its deep connection to nature and outdoor culture. In 2025, Invented. ReInvented. extended that vision to the world’s leading cities, showcasing how Salomon’s heritage continually adapts to new environments, practices and communities.

In 2026 Salomon opens new chapter with a campaign dedicated to the mindset that has always set it apart: innovation. Fully asserting its role as a force that shapes both sport and culture.

Guillaume Meyzenq, CEO, Salomon: “2026 marks a significant step in a journey we began in the French Alps 80 years ago.Our mission is to conn

In [29]:

import os
from google import genai

google_client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

conversation_history = []

def format_campaigns_as_context(results):
    context = ""
    for i, r in enumerate(results, 1):
        context += f"""
Campaign {i}:
- Title: {r['title']}
- Brand: {r['brand']}
- Industry: {r['industry']}
- Score: {r['score']}
- Description: {r['description']}
- URL: {r['url']}
---
"""
    return context

def chat(user_message):
    results = hybrid_search(user_message, campaigns, bm25, embeddings, model, top_k=10)
    context = format_campaigns_as_context(results)

    system_prompt = f"""You are a marketing campaign expert assistant.
You help users find and analyze advertising campaigns from 'campaigns.json'.
Answer based on the retrieved campaigns below. 
Answer questions by providing the campaign name, brand, industry, description, url to the campaign.
If the user asks something unrelated to the campaigns, politely redirect them.

Retrieved campaigns:
{context}"""

    conversation_history.append({
        "role": "user",
        "parts": [{"text": user_message}]
    })

    response = google_client.models.generate_content(
        model="models/gemini-2.5-flash",
        contents=conversation_history,
        config={
            "system_instruction": system_prompt,
            "max_output_tokens": 1000,
        }
    )

    assistant_message = response.text

    
    conversation_history.append({
        "role": "model",
        "parts": [{"text": assistant_message}]
    })

    return assistant_message


# Chat loop
print("Marketing Campaign Assistant (type 'quit' to exit)\n")
while True:
    user_input = input("You: ")
    if user_input.lower() in ["quit", ""]:
        break
    response = chat(user_input)
    print(f"\nAssistant: {response}\n")

Marketing Campaign Assistant (type 'quit' to exit)


Assistant: Here is the campaign related to Salomon:

**Campaign Name:** Shaping New Futures
**Brand:** Salomon
**Industry:** Sportswear
**Description:** On February 1, SALOMON launches Shaping New Futures, its latest global brand campaign, positioning innovation as the brand’s cultural engine: a way to shape sport, influence style, and design a modern mountain sport culture, while staying true to Salomon’s roots. The campaign reflects Salomon's belief that the future is not something to predict, but something to create, driven by curiosity, experimentation, bravery, creativity, and a constant desire to challenge boundaries.
**URL:** https://www.adsoftheworld.com/campaigns/shaping-new-futures



In [23]:
# Kiểm tra xem Solomon được lưu thế nào
for c in campaigns:
    if 'salomon' in str(c).lower():
        print(c)

{'id': 39, 'url': 'https://www.adsoftheworld.com/campaigns/shaping-new-futures', 'title': 'Shaping New Futures', 'brand': 'Salomon', 'agency': 'BBDO Paris', 'country': 'France', 'medium': 'Film', 'industry': 'Sportswear', 'description': 'On February 1, SALOMON launches Shaping New Futures, its latest global brand campaign, positioning innovation as the brand’s cultural engine: a way to shape sport, influence style, and design a modern mountain sport culture, while staying true to Salomon’s roots.\n\nOver the past three years, SALOMON has followed a clear and purposeful brand trajectory. In 2024, Welcome Back to Earth reaffirmed its deep connection to nature and outdoor culture. In 2025, Invented. ReInvented. extended that vision to the world’s leading cities, showcasing how Salomon’s heritage continually adapts to new environments, practices and communities.\n\nIn 2026 Salomon opens new chapter with a campaign dedicated to the mindset that has always set it apart: innovation. Fully ass

In [20]:
# Kiểm tra số lượng campaigns
print(len(campaigns))

424


In [30]:
import streamlit as st
import os
from google import genai

# --- Giữ nguyên các hàm cũ ---
# (copy toàn bộ: load campaigns, build bm25, embeddings, hybrid_search, format_campaigns_as_context)

# --- Streamlit UI ---
st.title("Marketing Campaign Assistant")

# Khởi tạo google client
google_client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

# Lưu conversation history trong session
if "conversation_history" not in st.session_state:
    st.session_state.conversation_history = []

if "messages" not in st.session_state:
    st.session_state.messages = []

# Hiển thị lịch sử chat
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# Input từ user
user_input = st.chat_input("Ask about campaigns...")

if user_input:
    # Hiển thị tin nhắn user
    with st.chat_message("user"):
        st.write(user_input)
    st.session_state.messages.append({"role": "user", "content": user_input})

    # Gọi hàm chat
    results = hybrid_search(user_input, campaigns, bm25, embeddings, model, top_k=10)
    context = format_campaigns_as_context(results)

    system_prompt = f"""You are a marketing campaign expert assistant...
Retrieved campaigns:
{context}"""

    st.session_state.conversation_history.append({
        "role": "user",
        "parts": [{"text": user_input}]
    })

    response = google_client.models.generate_content(
        model="gemini-2.0-flash",
        contents=st.session_state.conversation_history,
        config={
            "system_instruction": system_prompt,
            "max_output_tokens": 1000,
        }
    )

    assistant_message = response.text
    st.session_state.conversation_history.append({
        "role": "model",
        "parts": [{"text": assistant_message}]
    })

    # Hiển thị tin nhắn assistant
    with st.chat_message("assistant"):
        st.write(assistant_message)
    st.session_state.messages.append({"role": "assistant", "content": assistant_message})


2026-03-26 23:57:31.183 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 23:57:31.432 
  command:

    streamlit run c:\Users\khanh\ai-engineering-fordham\.venv\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-03-26 23:57:31.434 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 23:57:31.436 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 23:57:33.354 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 23:57:33.355 Session state does not function when running a script without `streamlit run`
2026-03-26 23:57:33.356 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 23:57:33.357 Thread 'MainThread': missing

In [1]:
# Đoạn code kiểm tra model khả dụng
models = google_client.models.list()
for m in models:
    print(f"Model Name: {m.name} - Supported Methods: {m.supported_methods}")
    # Hoặc hiển thị lên giao diện Streamlit
    st.write(f"Model: {m.name}")

NameError: name 'google_client' is not defined

In [5]:
import json
import os
import re

with open("project/campaigns.json", "r", encoding="utf-8") as f:
      campaigns = json.load(f)

print(f"Loaded {len(campaigns)} campaigns")

Loaded 424 campaigns


In [6]:
patterns = {
    "AOTW auto-sentence": r"This\s+professional campaign titled",
    "Newsletter CTA":     r"Don't miss out",
    "Credit line":        r"Client\s*:",
}

for pattern_name, pattern in patterns.items():
    matches = [c for c in campaigns if re.search(pattern, c.get("description", ""))]
    print(f"{pattern_name}: found in {len(matches)}/{len(campaigns)} campaigns")
    if matches:
        sample = matches[0]
        print(f"  Example: [{sample['id']}] {sample['title'][:50]}")
        snippet = re.search(pattern, sample['description']).group(0)
        print(f"  Matched: '{snippet}'\n")

AOTW auto-sentence: found in 283/424 campaigns
  Example: [2] THE KAPRAO CRIMINALS
  Matched: 'This  professional campaign titled'

Newsletter CTA: found in 180/424 campaigns
  Example: [2] THE KAPRAO CRIMINALS
  Matched: 'Don't miss out'

Credit line: found in 41/424 campaigns
  Example: [2] THE KAPRAO CRIMINALS
  Matched: 'Client :'



In [8]:
# Tìm campaigns mà description CHỈ có junk, không có nội dung thật
import re

def has_real_description(desc):
    """
    Xóa tạm các junk patterns, xem còn lại gì không
    """
    text = re.sub(r"This\s+professional campaign titled.*?media asset[s]?\.", "", desc, flags=re.DOTALL)
    text = re.sub(r"Don't miss out\..*?confidential\.", "", text, flags=re.DOTALL)
    text = re.sub(r"Client\s*:.*", "", text, flags=re.DOTALL)
    text = text.strip()
    return len(text) > 50  # còn lại ít nhất 50 chars = có nội dung thật

no_real_desc = [c for c in campaigns if not has_real_description(c.get("description", ""))]
has_real_desc = [c for c in campaigns if has_real_description(c.get("description", ""))]

print(f"has real desc: {len(has_real_desc)}/424")
print(f"only junk/empty: {len(no_real_desc)}/424")

# Xem vài ví dụ không có description thật
print("\n--- Examples of campaigns without original description ---")
for c in no_real_desc[:3]:
    print(f"\n[{c['id']}] {c['title'][:50]}")
    print(f"original desc: {c['description'][:200]}")

print("\n--- Examples of campaigns with original description ---")
for c in has_real_desc[:3]:
    print(f"\n[{c['id']}] {c['title'][:50]}")
    print(f"original desc: {c['description'][:200]}")

has real desc: 421/424
only junk/empty: 3/424

--- Examples of campaigns without original description ---

[129] Drink Different
original desc: This  professional campaign titled 'Drink Different' was published in Australia and New Zealand in January, 2025. It was created for the brand: Ocean Spray, by ad agency: The Reactor. This OOH Outdoor

[232] Anthem
original desc: This  professional campaign titled 'Anthem' was published in United States in February, 2026. It was created for the brand: Scientology, by ad agency: Scientology Media Productions. This Film medium c

[420] A New Way
original desc: This  professional campaign titled 'A New Way' was published in United States in February, 2026. It was created for the brand: Novo Nordisk, by ad agency: M+C Saatchi Group. This Film medium campaign 

--- Examples of campaigns with original description ---

[1] Rebranding and communication for Selfish, reinforc
original desc: Rebranding and communication for Selfish, reinforcing its positi

In [9]:
import re

# Take campaign [2] description
c2 = [c for c in campaigns if c["id"] == 2][0]
desc = c2["description"]

print("=== ORIGINAL ===")
print(repr(desc))

# Apply cleaning step by step
text = re.sub(r"This\s+professional campaign titled.*?media asset[s]?\.", "", desc, flags=re.DOTALL)
print("\n=== AFTER removing auto-sentence ===")
print(repr(text))

text = re.sub(r"Don't miss out\..*?confidential\.", "", text, flags=re.DOTALL)
print("\n=== AFTER removing newsletter CTA ===")
print(repr(text))

text = re.sub(r"Client\s*:.*", "", text, flags=re.DOTALL)
print("\n=== AFTER removing credit line ===")
print(repr(text))

print(f"\n=== FINAL length: {len(text.strip())} chars ===")

=== ORIGINAL ===
"‘Kaprao’ the dish the whole nation takes seriously.\n\nThis  professional campaign titled 'THE KAPRAO CRIMINALS' was published in Thailand in February, 2026. It was created for the brand: KFC Thailand, by ad agency: Wolf BKK. This Film medium campaign is related to the Food industry and contains 1 media asset. It was submitted about 1 month ago.\n\nClient : KFC ThailandAgency : Wolf BKKProduction House : Suneta House\n\nDon't miss out. Receive our free weekly newsletter to learn about the best creative work from all around the globe. We're keeping your email safe and confidential."

=== AFTER removing auto-sentence ===
"‘Kaprao’ the dish the whole nation takes seriously.\n\n It was submitted about 1 month ago.\n\nClient : KFC ThailandAgency : Wolf BKKProduction House : Suneta House\n\nDon't miss out. Receive our free weekly newsletter to learn about the best creative work from all around the globe. We're keeping your email safe and confidential."

=== AFTER removing n

In [13]:
import re
import json

def clean_description(text: str) -> str:
    text = re.sub(r"This\s+professional campaign titled.*?media asset[s]?\.", "", text, flags=re.DOTALL)
    text = re.sub(r"It was submitted .{3,30} ago\.", "", text)
    text = re.sub(r"Client\s*:.*", "", text, flags=re.DOTALL)
    text = re.sub(r"Don't miss out\..*?confidential\.", "", text, flags=re.DOTALL)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

# Load original
with open("project/campaigns.json", "r", encoding="utf-8") as f:
    campaigns = json.load(f)

# Clean descriptions in place
for c in campaigns:
    c["description"] = clean_description(c.get("description", ""))

# Save to new file — original untouched
with open("project/campaigns_cleaned.json", "w", encoding="utf-8") as f:
    json.dump(campaigns, f, ensure_ascii=False, indent=2)

print(f"Done. {len(campaigns)} campaigns cleaned → campaigns_cleaned.json")

Done. 424 campaigns cleaned → campaigns_cleaned.json


In [20]:
c2 = [c for c in campaigns if c["id"] == 2][0]
print(len(c2["description"]))  # 51
print(51 < 70)                 # should print True

51
True


In [21]:
short = [c for c in campaigns if len(c["description"]) < 70]
print(f"Count: {len(short)}")
for c in short:
    print(f"[{c['id']}] len={len(c['description'])} — {repr(c['description'])}")

Count: 9
[2] len=51 — '‘Kaprao’ the dish the whole nation takes seriously.'
[67] len=65 — 'It was submitted about 2 months ago by Mr: Greg of Romance Films.'
[70] len=63 — 'No matter how life unfolds, family remains our truest blessings'
[129] len=0 — ''
[232] len=0 — ''
[236] len=48 — 'Shaping your ideas into apps that work your way.'
[418] len=50 — 'A bread‑taking story only Morgan Freeman can tell.'
[419] len=49 — 'Lay’s opens a non-existent restaurant on UberEats'
[420] len=0 — ''


In [23]:
def clean_description(text: str) -> str:
    text = re.sub(r"This\s+professional campaign titled.*?media asset[s]?\.", "", text, flags=re.DOTALL)
    text = re.sub(r"It was submitted .+?ago[^.]*\.", "", text)  # fixed
    text = re.sub(r"Client\s*:.*", "", text, flags=re.DOTALL)
    text = re.sub(r"Don't miss out\..*?confidential\.", "", text, flags=re.DOTALL)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

# Re-apply to original campaigns
with open("project/campaigns.json", "r", encoding="utf-8") as f:
    campaigns = json.load(f)

for c in campaigns:
    c["description"] = clean_description(c.get("description", ""))

with open("project/campaigns_cleaned.json", "w", encoding="utf-8") as f:
    json.dump(campaigns, f, ensure_ascii=False, indent=2)

print("Done. Re-cleaned and saved.")

Done. Re-cleaned and saved.


In [24]:
c67 = [c for c in campaigns if c["id"] == 67][0]
print(repr(c67["description"]))  # should be empty or just the real content

''


In [2]:
import json

with open("project/campaigns_cleaned.json", "r", encoding="utf-8") as f:
    old_campaigns = json.load(f)

new_campaigns = []
for i, c in enumerate(old_campaigns):
    new = {
        "metadata": {
            "id":             i + 1,
            "url":            c.get("url", ""),
            "title":          c.get("title", ""),
            "brand":          c.get("brand", ""),
            "agency":         c.get("agency", ""),
            "industry":       c.get("industry", ""),
            "country":        c.get("country", ""),
            "medium":         c.get("medium", ""),
            "tags":           c.get("tags", []),
            "published_date": c.get("published_date", ""),
        },
        "content": {
            "description":   c.get("description", ""),
            "thumbnail_url": c.get("thumbnail_url", ""),
            "media_count":   c.get("media_count", 0),
        },
        "ai_enrichment": {
            "concept_summary": "",
            "target_audience": "",
            "tactics":         []
        },
        "system": {
            "crawled_at":  c.get("crawled_at", ""),
            "is_enriched": False
        }
    }
    new_campaigns.append(new)

with open("project/campaigns_v2.json", "w", encoding="utf-8") as f:
    json.dump(new_campaigns, f, ensure_ascii=False, indent=2)

print(f"Done. {len(new_campaigns)} campaigns migrated → campaigns_v2.json")

Done. 424 campaigns migrated → campaigns_v2.json


In [9]:
import json
import time
import os
import re
from google import genai
from google.genai import types

# Load file FIRST
with open("project/campaigns_v2.json", "r", encoding="utf-8") as f:
    campaigns = json.load(f)

client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

ENRICH_PROMPT = """
Based on given campaign information, analyze it and return ONLY a valid JSON object, no markdown, no backticks:

Campaign: "{title}" by {brand}, medium: {medium}, industry: {industry}
Description: {description}

{{
  "concept_summary": "1-2 sentences describing the core creative idea",
  "target_audience": "Who this campaign targets, be specific e.g. Thai street food lovers aged 18-35",
  "execution_tactics":"1-2 sentences, focus on channels, activations e.g. Hero film on YouTube + TV",
  "objective": "1 sentence describing objective of the campaign? e.g. Brand awareness, sales, engagement, etc."
}}
"""

def enrich_campaign(c):
    meta = c["metadata"]
    desc = c["content"]["description"]
    
    prompt = ENRICH_PROMPT.format(
        title=meta.get("title", ""),
        brand=meta.get("brand", ""),
        industry=meta.get("industry", ""),
        medium=meta.get("medium", ""),
        description=desc[:800]
    )

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[{"role": "user", "parts": [{"text": prompt}]}],
        config=types.GenerateContentConfig(max_output_tokens=1500),
    )
    
    raw = response.text.strip()
    print(f"=== RAW RESPONSE ===")
    print(repr(raw))
    print(f"Length: {len(raw)}")
    print(f"Finish reason: {response.candidates[0].finish_reason}")
    print(f"====================")
    
    if "```" in raw:
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    
    return json.loads(raw.strip())

# Test on 2 campaigns only
total = len(campaigns)
enriched_count = 0

for i, c in enumerate(campaigns):
    if c["system"]["is_enriched"]:
        print(f"[{i+1}/{total}] Skipping: {c['metadata']['title'][:50]}")
        continue

    try:
        enrichment = enrich_campaign(c)
        c["ai_enrichment"] = enrichment
        c["system"]["is_enriched"] = True
        enriched_count += 1
        print(f"[{i+1}/{total}] ✓ {c['metadata']['title'][:50]}")
        #print(json.dumps(enrichment, indent=2, ensure_ascii=False))
    except Exception as e:
        print(f"[{i+1}/{total}] ✗ Failed: {c['metadata']['title'][:50]} — {e}")
    
    if (i + 1) % 20 == 0:
        with open("project/campaigns_v2.json", "w", encoding="utf-8") as f:
            json.dump(campaigns, f, ensure_ascii=False, indent=2)
        print(f"  [checkpoint: {enriched_count} enriched so far]")

    time.sleep(1)

with open("project/campaigns_v2.json", "w", encoding="utf-8") as f:
    json.dump(campaigns, f, ensure_ascii=False, indent=2)

print(f"\nDone. {enriched_count}/{total} campaigns enriched.")

[1/424] Skipping: Rebranding and communication for Selfish, reinforc
[2/424] Skipping: THE KAPRAO CRIMINALS
[3/424] Skipping: Take Yourself Funny For Money
[4/424] Skipping: Every Journey Starts Somewhere
[5/424] Skipping: ScrollSticks
[6/424] Skipping: Forever Sweethearts
[7/424] Skipping: 2026 Valentine’s Day campaign
[8/424] Skipping: Chew Bold
[9/424] Skipping: 纸为你 (Crafted With Love)
[10/424] Skipping: The Tracktaste
[11/424] Skipping: Smell Like Your Ex(foliating Body Wash)
[12/424] Skipping: Wild Transfers - BBVA
[13/424] Skipping: This is the Good Stuff
[14/424] Skipping: The Iron Standard
[15/424] Skipping: Dirt Road
[16/424] Skipping: Who was the victim?
[17/424] Skipping: Fans Have More Friends
[18/424] Skipping: Juicy Gummy Clusters
[19/424] Skipping: 'Someday Starts Today
[20/424] Skipping: GP Access – Tap it, type it and we’ll take care of
[21/424] Skipping: Happy Valentine’s DA.I.
[22/424] Skipping: chief sleep officer
[23/424] Skipping: Label created by the wine itself


In [10]:
with open("project/campaigns_v2.json", "r", encoding="utf-8") as f:
    campaigns = json.load(f)

enriched = [c for c in campaigns if c["system"]["is_enriched"]]
not_enriched = [c for c in campaigns if not c["system"]["is_enriched"]]

print(f"Enriched: {len(enriched)}/424")
print(f"Not enriched (failed or MAX_TOKENS): {not_enriched}")

Enriched: 424/424
Not enriched (failed or MAX_TOKENS): []


In [11]:
import json

with open("project/campaigns_v2.json", "r", encoding="utf-8") as f:
    campaigns = json.load(f)

# Check 3 random campaigns
import random
samples = random.sample(campaigns, 3)
for c in samples:
    print(f"\n=== {c['metadata']['title'][:50]} ===")
    print(json.dumps(c["ai_enrichment"], indent=2, ensure_ascii=False))


=== And Then Came The Egg ===
{
  "concept_summary": "The campaign creatively answers the age-old 'chicken or egg' question by asserting that the land, dedicated farmers, and ethical hen-raising practices come first, highlighting Vital Farms' commitment to quality and responsible sourcing.",
  "target_audience": "Conscious consumers who prioritize ethically sourced food, animal welfare, and support brands committed to sustainable and humane farming practices.",
  "execution_tactics": "Launching a 60-second film as Vital Farms' first-ever Big Game ad at 5 AM, specifically timed to coincide with the early start of their farmers' workday.",
  "objective": "To build brand awareness, reinforce Vital Farms' commitment to ethical farming and quality, and differentiate the brand through its core values."
}

=== Choretasia ===
{
  "concept_summary": "The campaign promotes BCAA's new Task Marketplace by re-imagining Disney's Fantasia, using practical effects and puppetry to bring dancing chore 